# PatchCore EfficientNet-B0 Notebook (x224)

This notebook works with the checked-in EfficientNet-B0 PatchCore run for the `x224` benchmark branch.

## Submission Context

- Dataset notebook: `data/dataset/x224/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x224/benchmark_50k_5pct/data_config.toml`
- Metadata CSV: `data/processed/x224/wm811k/metadata_50k_5pct.csv`
- Artifact root: `experiments/anomaly_detection/patchcore/efficientnet_b0/x224/main/artifacts/patchcore_efficientnet_b0_5pct`
- Mode: artifact-backed evaluation using the saved local checkpoint and result files
- Checkpoint status: checked-in local checkpoint available under `checkpoints/`

## Imports and Saved Results

This cell loads the saved summary, score CSVs, and local metadata needed for review plots and defect analysis.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm import tqdm

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.evaluation import summarize_threshold_metrics

## Metrics and Plots

These cells recreate the main evaluation figures from the saved score CSVs and store them under `plots/`.

In [ ]:
metrics_df = pd.DataFrame(
    [
        {"metric": "precision", "value": metrics["precision"]},
        {"metric": "recall", "value": metrics["recall"]},
        {"metric": "f1", "value": metrics["f1"]},
        {"metric": "auroc", "value": metrics["auroc"]},
        {"metric": "auprc", "value": metrics["auprc"]},
        {"metric": "threshold", "value": threshold},
    ]
)
confusion_df = pd.DataFrame(metrics["confusion_matrix"], index=["true_normal", "true_anomaly"], columns=["pred_normal", "pred_anomaly"])
display(metrics_df)
display(confusion_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(val_scores_df["score"], bins=30, alpha=0.85, color="#264653")
axes[0].axvline(threshold, color="red", linestyle="--")
axes[0].set_title("Validation Normal Score Distribution")
axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 0]["score"], bins=30, alpha=0.7, label="normal", color="#577590")
axes[1].hist(test_scores_df[test_scores_df["is_anomaly"] == 1]["score"], bins=30, alpha=0.7, label="anomaly", color="#e76f51")
axes[1].axvline(threshold, color="red", linestyle="--")
axes[1].set_title("Test Score Distribution")
axes[1].legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / "score_distribution.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_sweep_df.iloc[:, 0], threshold_sweep_df["precision"], label="precision")
ax.plot(threshold_sweep_df.iloc[:, 0], threshold_sweep_df["recall"], label="recall")
ax.plot(threshold_sweep_df.iloc[:, 0], threshold_sweep_df["f1"], label="f1")
ax.set_title("Threshold Sweep")
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / "threshold_sweep.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

## Defect Analysis

This cell recomputes failure analysis from the saved test scores and local metadata, then saves the resulting tables and plot.

In [ ]:
analysis_df = test_metadata.copy()
analysis_df["score"] = test_scores_df.reset_index(drop=True)["score"]
analysis_df["predicted_anomaly"] = (analysis_df["score"] > threshold).astype(int)
analysis_df["error_type"] = "tn"
analysis_df.loc[(analysis_df["is_anomaly"] == 0) & (analysis_df["predicted_anomaly"] == 1), "error_type"] = "fp"
analysis_df.loc[(analysis_df["is_anomaly"] == 1) & (analysis_df["predicted_anomaly"] == 0), "error_type"] = "fn"
analysis_df.loc[(analysis_df["is_anomaly"] == 1) & (analysis_df["predicted_anomaly"] == 1), "error_type"] = "tp"

error_summary_df = (
    analysis_df.groupby("error_type")
    .agg(count=("error_type", "size"), mean_score=("score", "mean"))
    .reindex(["tp", "fn", "fp", "tn"])
)
defect_recall_df = (
    analysis_df[analysis_df["is_anomaly"] == 1]
    .groupby("defect_type")
    .agg(count=("defect_type", "size"), detected=("predicted_anomaly", "sum"), mean_score=("score", "mean"))
    .sort_values(["detected", "count"], ascending=[False, False])
)
defect_recall_df["recall"] = defect_recall_df["detected"] / defect_recall_df["count"]

analysis_df.to_csv(EVAL_DIR / "analysis_with_predictions.csv", index=False)
error_summary_df.reset_index().to_csv(EVAL_DIR / "error_summary.csv", index=False)
defect_recall_df.reset_index().to_csv(EVAL_DIR / "defect_recall.csv", index=False)

top_defects_df = defect_recall_df.head(10).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(error_summary_df.index.astype(str), error_summary_df["count"], color="#e76f51")
axes[0].set_title("Prediction Outcome Counts")
axes[1].barh(top_defects_df["defect_type"], top_defects_df["recall"], color="#8ab17d")
axes[1].set_xlim(0.0, 1.0)
axes[1].invert_yaxis()
axes[1].set_title("Top Defect-Type Recall")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "defect_breakdown.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

display(error_summary_df)
display(defect_recall_df)

## UMAP: Resolution Effect (x64 vs x224)

Extracts wafer-level EfficientNet-B0 embeddings at two input resolutions using the
saved checkpoint, runs UMAP on both, and saves a side-by-side comparison plot.

**Run time:** ~3 minutes on CPU. No GPU required.

**Outputs:**
- `experiments/anomaly_detection/patchcore/efficientnet_b0/x64/umap_points.csv`
- `experiments/anomaly_detection/patchcore/efficientnet_b0/x224/umap_points.csv`
- `experiments/anomaly_detection/report_figures/artifacts/plots/umap_resolution_comparison_effb0.png`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import umap as umap_lib
from tqdm import tqdm

# -- Paths ---------------------------------------------------------------------
CHECKPOINT  = REPO_ROOT / "experiments/anomaly_detection/patchcore/efficientnet_b0/x224/main/artifacts/checkpoints/best_model.pt"
META_X64    = REPO_ROOT / "data/processed/x64/wm811k/metadata_50k_5pct.csv"
META_X224   = REPO_ROOT / "data/processed/x224/wm811k/metadata_50k_5pct.csv"
OUT_X64     = REPO_ROOT / "experiments/anomaly_detection/patchcore/efficientnet_b0/x64/umap_points.csv"
OUT_X224    = REPO_ROOT / "experiments/anomaly_detection/patchcore/efficientnet_b0/x224/umap_points.csv"
OUT_PLOT    = REPO_ROOT / "experiments/anomaly_detection/report_figures/artifacts/plots/umap_resolution_comparison_effb0.png"
for p in [OUT_X64.parent, OUT_X224.parent, OUT_PLOT.parent]:
    p.mkdir(parents=True, exist_ok=True)

MID_IDX = 3; DEEP_IDX = 6; EMBED_DIM = 512; SZ = 224

# -- Feature extractor (exact architecture from archive notebook) ---------------
class _Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.features = backbone.features
        with torch.inference_mode():
            dummy = torch.zeros(1, 3, SZ, SZ); x = dummy; f_mid = f_deep = None
            for i, block in enumerate(self.features):
                x = block(x)
                if i == MID_IDX:  f_mid  = x
                if i == DEEP_IDX: f_deep = x
        self.in_dim = f_mid.shape[1] + f_deep.shape[1]
        self.proj = nn.Linear(self.in_dim, EMBED_DIM, bias=False)
        for p in list(self.features.parameters()) + list(self.proj.parameters()):
            p.requires_grad_(False)

    def forward(self, x):
        f_mid = f_deep = None; h = x
        for i, block in enumerate(self.features):
            h = block(h)
            if i == MID_IDX:  f_mid  = h
            if i == DEEP_IDX: f_deep = h
        f_d  = F.interpolate(f_deep, size=f_mid.shape[-2:], mode="bilinear", align_corners=False)
        cat  = torch.cat([f_mid, f_d], dim=1)
        flat = cat.permute(0, 2, 3, 1).reshape(x.shape[0], -1, self.in_dim)
        return self.proj(flat).mean(dim=1)   # (B, 512) wafer-level mean

model = _Extractor()
ckpt  = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
model.proj.load_state_dict({k.replace("proj.",""): v for k, v in ckpt["model_state_dict"].items() if "proj" in k})
model.eval()
print(f"Checkpoint loaded  memory_bank={ckpt['memory_bank'].shape}")

# -- Load metadata -------------------------------------------------------------
def load_meta(path):
    m = pd.read_csv(path)
    t = m[m["split"] == "test"].reset_index(drop=True)
    t["label"] = t["failure_type"].fillna("Normal")
    t.loc[t["label"] == "none", "label"] = "Normal"
    return t

meta64  = load_meta(META_X64)
meta224 = load_meta(META_X224)

def load_wafer(array_path):
    arr    = np.load(REPO_ROOT / array_path)
    t      = torch.from_numpy(arr).float()
    states = torch.clamp(torch.round(t * 2.0), 0, 2).long().squeeze(0)
    oh     = F.one_hot(states, 3).permute(2, 0, 1).float()
    if oh.shape[-1] != SZ:
        oh = F.interpolate(oh.unsqueeze(0), (SZ, SZ), mode="bilinear", align_corners=False).squeeze(0)
    return oh

# -- Extract embeddings --------------------------------------------------------
BATCH = 32
embs64, embs224 = [], []
for start in tqdm(range(0, len(meta64), BATCH), desc="Extracting embeddings"):
    b64  = meta64.iloc[start:start+BATCH]
    b224 = meta224.iloc[start:start+BATCH]
    imgs64  = [load_wafer(r["array_path"]) for _, r in b64.iterrows()]
    imgs224 = [load_wafer(r["array_path"]) for _, r in b224.iterrows()]
    with torch.no_grad():
        embs64.append( model(torch.stack(imgs64)).numpy())
        embs224.append(model(torch.stack(imgs224)).numpy())

embs64  = np.concatenate(embs64,  axis=0)
embs224 = np.concatenate(embs224, axis=0)
print(f"Embeddings: x64={embs64.shape}  x224={embs224.shape}")

# -- Run UMAP ------------------------------------------------------------------
print("Running UMAP x64 ...")
c64  = umap_lib.UMAP(n_neighbors=20, min_dist=0.1, random_state=42).fit_transform(embs64)
print("Running UMAP x224 ...")
c224 = umap_lib.UMAP(n_neighbors=20, min_dist=0.1, random_state=42).fit_transform(embs224)

# -- Save CSVs -----------------------------------------------------------------
for coords, meta, out_path, include_score in [
    (c64,  meta64,  OUT_X64,  False),
    (c224, meta224, OUT_X224, True),
]:
    df = pd.DataFrame({
        "umap_1": coords[:, 0], "umap_2": coords[:, 1],
        "is_anomaly": (meta["label"] != "Normal").astype(int).values,
        "label": meta["label"].values,
    })
    if include_score:
        df["score"] = test_scores_df["score"].values
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

# -- Plot (4 rows x 1 column: vertical stack) ----------------------------------
DTYPES = ["Edge-Ring","Edge-Loc","Center","Loc","Scratch","Donut","Random","Near-full"]
DC = {
    "Normal":    ("#d0d0d0", 3,  0.12),
    "Edge-Ring": ("#e41a1c", 60, 0.90), "Edge-Loc":  ("#ff7f00", 60, 0.90),
    "Center":    ("#4daf4a", 60, 0.90), "Loc":       ("#984ea3", 60, 0.90),
    "Scratch":   ("#a65628", 80, 0.95), "Donut":     ("#f781bf", 60, 0.90),
    "Random":    ("#377eb8", 60, 0.90), "Near-full": ("#ffff33", 70, 0.95),
}

df64  = pd.read_csv(OUT_X64)
df224 = pd.read_csv(OUT_X224)

fig, axes = plt.subplots(4, 1, figsize=(8, 14))
fig.suptitle(
    "Resolution Effect: EfficientNet-B0 PatchCore\n"
    "Top 2 panels: 64x64 preprocessing (F1=0.467)   Bottom 2 panels: Native 224x224 (F1=0.544)",
    fontsize=12, fontweight="bold",
)

legend_handles = []
row_idx = 0
for ci, (df, title) in enumerate([(df64, "64x64"), (df224, "224x224")]):
    ax_s = axes[row_idx]
    ax_c = axes[row_idx + 1]
    ux, uy = df["umap_1"].values, df["umap_2"].values

    # Left plot: defect type
    col, ms, al = DC["Normal"]
    ax_s.scatter(ux[df["label"] == "Normal"], uy[df["label"] == "Normal"],
                 c=col, s=ms, alpha=al, linewidths=0, zorder=1)
    for dt in DTYPES:
        col, ms, al = DC[dt]; mask = df["label"] == dt
        if mask.sum() > 0:
            ax_s.scatter(ux[mask], uy[mask], c=col, s=ms, alpha=al,
                         linewidths=0.4, edgecolors="white", zorder=3)
            if ci == 0:
                legend_handles.append(mpatches.Patch(color=col, label=f"{dt} ({mask.sum()})"))
    ax_s.set_title(f"{title}: Defect type", fontsize=11, fontweight="bold")
    ax_s.set_xticks([]); ax_s.set_yticks([])

    # Right plot: anomaly score
    if "score" in df.columns:
        sv  = df["score"].values
        sc2 = ax_c.scatter(ux, uy, c=sv, cmap="YlOrRd", s=4, alpha=0.6, linewidths=0,
                           vmin=np.percentile(sv, 2), vmax=np.percentile(sv, 98))
        plt.colorbar(sc2, ax=ax_c, fraction=0.04, pad=0.02, label="Anomaly score")
    else:
        ax_c.scatter(ux, uy, c=df["is_anomaly"].values, cmap="RdGy_r",
                     s=4, alpha=0.5, linewidths=0, vmin=0, vmax=1)
    ax_c.set_title(f"{title}: Anomaly score", fontsize=11, fontweight="bold")
    ax_c.set_xticks([]); ax_c.set_yticks([])

    row_idx += 2

fig.legend(handles=legend_handles, loc="lower center", ncol=4, fontsize=9,
           framealpha=0.9, bbox_to_anchor=(0.5, -0.02),
           title="Defect type  (normals shown as grey background)", title_fontsize=8)
fig.text(0.5, -0.08,
         "Same EfficientNet-B0 backbone and PatchCore scoring. Only preprocessing resolution changes.",
         ha="center", fontsize=9, style="italic", color="#555")
plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.savefig(OUT_PLOT, dpi=150, bbox_inches="tight")
plt.close()
print(f"Plot saved: {OUT_PLOT}")